# Training Face Recognition - Missing Person Detection System

## Using InsightFace (ArcFace ResNet100) - 512-d Embeddings

### Notebook objectives:
1. **Load and inspect images** from the missing person database
2. **Generate face embeddings** (512-d vectors) for each person
3. **Evaluate embedding quality** (intra/inter-person cosine similarity)
4. **Tune recognition threshold** (cosine similarity threshold)
5. **Save embeddings database** for use in the detection pipeline
6. **(Optional) Train SVM/KNN classifier** on embeddings

### Database directory structure:
```
missing_persons_db/
├── person_001/
│   ├── name.txt          # Person's name (e.g. "John Doe")
│   ├── photo1.jpg
│   ├── photo2.jpg
│   └── photo3.jpg
├── person_002/
│   ├── name.txt
│   └── photo1.jpg
└── ...
```

## 1. Setup & Imports

In [ ]:
import os
import sys
import pickle
import numpy as np
import cv2
import matplotlib.pyplot as plt
from itertools import combinations
from insightface.app import FaceAnalysis

# Add root directory to path
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import config
from utils.face_recognizer import FaceRecognizer

# Check environment
print("=== Environment Check ===")
print(f"numpy version: {np.__version__}")
print(f"opencv version: {cv2.__version__}")
print(f"Database dir: {config.MISSING_PERSONS_DB_DIR}")
print(f"Embeddings file: {config.EMBEDDINGS_FILE}")

# Initialize InsightFace
print("\nInitializing InsightFace (ArcFace)...")
app = FaceAnalysis(name=config.INSIGHTFACE_MODEL, providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
print("InsightFace ready!")

# Check database directory
if os.path.exists(config.MISSING_PERSONS_DB_DIR):
    person_dirs = [d for d in os.listdir(config.MISSING_PERSONS_DB_DIR)
                   if os.path.isdir(os.path.join(config.MISSING_PERSONS_DB_DIR, d))
                   and d.startswith("person_")]
    print(f"\nFound {len(person_dirs)} person directories in database")
else:
    print(f"\nDatabase directory NOT FOUND!")
    print(f"mkdir -p {config.MISSING_PERSONS_DB_DIR}/person_001")

## 2. Load and Display Images from Database

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_person_data(db_dir):
    """Load all person data from database directory."""
    persons = []
    person_dirs = sorted([
        d for d in os.listdir(db_dir)
        if os.path.isdir(os.path.join(db_dir, d)) and d.startswith("person_")
    ])

    for person_dir in person_dirs:
        person_path = os.path.join(db_dir, person_dir)

        # Read name
        name_file = os.path.join(person_path, "name.txt")
        if os.path.exists(name_file):
            with open(name_file, "r", encoding="utf-8") as f:
                name = f.read().strip()
        else:
            name = person_dir

        # Get image list
        image_files = sorted([
            os.path.join(person_path, f)
            for f in os.listdir(person_path)
            if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS
        ])

        persons.append({
            "person_id": person_dir,
            "name": name,
            "image_files": image_files,
            "dir": person_path
        })

    return persons

# Load data
persons_data = load_person_data(config.MISSING_PERSONS_DB_DIR)

print(f"Total persons in database: {len(persons_data)}\n")
for p in persons_data:
    print(f"  {p['person_id']}: {p['name']} ({len(p['image_files'])} images)")

# Display sample images
if persons_data:
    max_cols = min(5, max(len(p["image_files"]) for p in persons_data) if persons_data else 1)
    fig, axes = plt.subplots(len(persons_data), max_cols,
                              figsize=(4 * max_cols, 4 * len(persons_data)),
                              squeeze=False)

    for i, person in enumerate(persons_data):
        for j in range(max_cols):
            ax = axes[i][j]
            if j < len(person["image_files"]):
                img = cv2.imread(person["image_files"][j])
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_rgb)
                ax.set_title(f"{person['name']}\n{os.path.basename(person['image_files'][j])}", fontsize=9)
            ax.axis("off")

    plt.suptitle("Missing Person Database Images", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("\nDatabase is empty! Add images to missing_persons_db/")

## 3. Generate Face Embeddings (ArcFace 512-d)

Using InsightFace ArcFace ResNet100 to generate 512-d vectors for each face.
- Pre-trained on MS1MV2 (~5.8M images, 85K identities)
- More accurate than dlib 128-d (LFW: 99.77% vs 99.38%)

In [ ]:
def generate_embeddings(persons_data, face_app):
    """Generate face embeddings for all persons in the database."""
    database = []

    for person in persons_data:
        print(f"\n{'='*50}")
        print(f"[{person['person_id']}] {person['name']}")
        print(f"{'='*50}")

        embeddings = []
        failed_images = []

        for img_path in person["image_files"]:
            img_name = os.path.basename(img_path)
            print(f"  Processing: {img_name}...", end=" ")

            image = cv2.imread(img_path)
            if image is None:
                print("CANNOT READ IMAGE!")
                failed_images.append(img_name)
                continue

            faces = face_app.get(image)

            if len(faces) == 0:
                print("No face found!")
                failed_images.append(img_name)
                continue

            if len(faces) > 1:
                print(f"Found {len(faces)} faces, using largest...", end=" ")
                areas = [(f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]) for f in faces]
                best_face = faces[np.argmax(areas)]
            else:
                best_face = faces[0]

            embeddings.append(best_face.embedding)
            print(f"OK (embedding shape: {best_face.embedding.shape}, "
                  f"det_score: {best_face.det_score:.3f})")

        print(f"\n  Result: {len(embeddings)}/{len(person['image_files'])} images succeeded")
        if failed_images:
            print(f"  Failed: {failed_images}")

        if embeddings:
            database.append({
                "person_id": person["person_id"],
                "name": person["name"],
                "embeddings": embeddings,
                "failed_images": failed_images
            })
        else:
            print(f"  NO EMBEDDINGS for {person['name']}!")

    return database

# Generate embeddings
print("Starting embedding generation...\n")
database = generate_embeddings(persons_data, app)

# Summary
total_embeddings = sum(len(p["embeddings"]) for p in database)
print(f"\n{'='*50}")
print(f"SUMMARY: {len(database)} persons, {total_embeddings} embeddings")
print(f"{'='*50}")

## 4. Evaluate Embedding Quality

- **Intra-person similarity**: Cosine similarity between images of the SAME person (should be high, > 0.5)
- **Inter-person similarity**: Cosine similarity between DIFFERENT persons (should be low, < 0.3)

In [ ]:
def cosine_similarity(emb1, emb2):
    """Compute cosine similarity between two embeddings."""
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

def evaluate_embeddings(database):
    """Evaluate embedding quality using intra/inter-person cosine similarity."""

    if len(database) < 1:
        print("Need at least 1 person in database to evaluate!")
        return None, None

    print("=== EMBEDDING QUALITY EVALUATION ===\n")

    # 1. Intra-person similarities (same person)
    print("--- Intra-person Cosine Similarity (same person, should be > 0.5) ---")
    intra_sims = []
    for person in database:
        encs = person["embeddings"]
        if len(encs) < 2:
            print(f"  {person['name']}: Only 1 image, cannot compute intra-similarity")
            continue

        person_sims = []
        for (enc1, enc2) in combinations(encs, 2):
            sim = cosine_similarity(enc1, enc2)
            person_sims.append(sim)
            intra_sims.append(sim)

        avg = np.mean(person_sims)
        min_s = np.min(person_sims)
        status = "OK" if avg > 0.5 else "WARNING"
        print(f"  [{status}] {person['name']}: avg={avg:.3f}, min={min_s:.3f} ({len(person_sims)} pairs)")

    # 2. Inter-person similarities (different persons)
    print(f"\n--- Inter-person Cosine Similarity (different persons, should be < 0.3) ---")
    inter_sims = []
    if len(database) >= 2:
        for (p1, p2) in combinations(database, 2):
            pair_sims = []
            for enc1 in p1["embeddings"]:
                for enc2 in p2["embeddings"]:
                    sim = cosine_similarity(enc1, enc2)
                    pair_sims.append(sim)
                    inter_sims.append(sim)

            avg = np.mean(pair_sims)
            max_s = np.max(pair_sims)
            status = "OK" if max_s < 0.3 else "WARNING"
            print(f"  [{status}] {p1['name']} vs {p2['name']}: avg={avg:.3f}, max={max_s:.3f}")
    else:
        print("  Need >= 2 persons for inter-person similarity")

    # 3. Summary
    print(f"\n--- SUMMARY ---")
    if intra_sims:
        print(f"  Intra-person: mean={np.mean(intra_sims):.3f}, min={np.min(intra_sims):.3f}")
    if inter_sims:
        print(f"  Inter-person: mean={np.mean(inter_sims):.3f}, max={np.max(inter_sims):.3f}")

    if intra_sims and inter_sims:
        gap = np.min(intra_sims) - np.max(inter_sims)
        if gap > 0:
            print(f"  [OK] Separation gap: {gap:.3f} (good)")
        else:
            print(f"  [WARNING] OVERLAP: {gap:.3f} (may cause mismatches)")

    # Plot distribution
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    if intra_sims:
        ax.hist(intra_sims, bins=20, alpha=0.7, label="Intra-person (same)", color="blue")
    if inter_sims:
        ax.hist(inter_sims, bins=20, alpha=0.7, label="Inter-person (different)", color="red")
    ax.axvline(x=config.RECOGNITION_THRESHOLD, color="green", linestyle="--",
               label=f"Threshold = {config.RECOGNITION_THRESHOLD}")
    ax.set_xlabel("Cosine Similarity")
    ax.set_ylabel("Count")
    ax.set_title("Embedding Cosine Similarity Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return intra_sims, inter_sims

eval_results = evaluate_embeddings(database)

## 5. Tune Recognition Threshold

In [ ]:
def find_optimal_threshold(intra_sims, inter_sims):
    """Find the optimal cosine similarity threshold."""
    if not intra_sims or not inter_sims:
        print("Need both intra and inter similarities to find optimal threshold.")
        print(f"Using default threshold: {config.RECOGNITION_THRESHOLD}")
        return config.RECOGNITION_THRESHOLD

    thresholds = np.arange(0.1, 0.8, 0.02)
    best_threshold = config.RECOGNITION_THRESHOLD
    best_accuracy = 0

    results_list = []
    for thresh in thresholds:
        tp = np.sum(np.array(intra_sims) >= thresh)
        fn = len(intra_sims) - tp
        tn = np.sum(np.array(inter_sims) < thresh)
        fp = len(inter_sims) - tn

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0

        results_list.append({
            "threshold": thresh, "accuracy": accuracy,
            "tpr": tpr, "fpr": fpr, "precision": precision
        })

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = thresh

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    threshs = [r["threshold"] for r in results_list]
    ax1.plot(threshs, [r["accuracy"] for r in results_list], "b-", label="Accuracy", linewidth=2)
    ax1.plot(threshs, [r["tpr"] for r in results_list], "g--", label="True Positive Rate")
    ax1.plot(threshs, [r["precision"] for r in results_list], "r--", label="Precision")
    ax1.axvline(x=best_threshold, color="orange", linestyle=":", label=f"Best = {best_threshold:.2f}")
    ax1.set_xlabel("Threshold (Cosine Similarity)")
    ax1.set_ylabel("Score")
    ax1.set_title("Metrics by Threshold")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    fprs = [r["fpr"] for r in results_list]
    tprs = [r["tpr"] for r in results_list]
    ax2.plot(fprs, tprs, "b-", linewidth=2)
    ax2.plot([0, 1], [0, 1], "r--", alpha=0.5)
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*50}")
    print(f"OPTIMAL THRESHOLD: {best_threshold:.2f} (accuracy = {best_accuracy:.1%})")
    print(f"Current config threshold: {config.RECOGNITION_THRESHOLD}")
    if abs(best_threshold - config.RECOGNITION_THRESHOLD) > 0.05:
        print(f"\nSuggestion: Update RECOGNITION_THRESHOLD = {best_threshold:.2f} in config.py")
    print(f"{'='*50}")

    return best_threshold

if eval_results and eval_results[0] is not None:
    intra_sims, inter_sims = eval_results
    optimal_threshold = find_optimal_threshold(intra_sims, inter_sims)
else:
    print("No evaluation data available. Using default threshold.")
    optimal_threshold = config.RECOGNITION_THRESHOLD

## 6. Save Embeddings Database

In [ ]:
# Prepare database for saving
save_database = []
for person in database:
    save_database.append({
        "person_id": person["person_id"],
        "name": person["name"],
        "embeddings": person["embeddings"]
    })

# Save pickle file
output_path = config.EMBEDDINGS_FILE
with open(output_path, "wb") as f:
    pickle.dump(save_database, f)

# Confirm
file_size = os.path.getsize(output_path) / 1024  # KB
print(f"Embeddings database saved!")
print(f"   File: {output_path}")
print(f"   Size: {file_size:.1f} KB")
print(f"   Persons: {len(save_database)}")
print(f"   Total embeddings: {sum(len(p['embeddings']) for p in save_database)}")
print(f"\nYou can now run:")
print(f"   python detect_missing_person.py --video path/to/video.mp4")

## 7. (Optional) Train SVM/KNN Classifier on Embeddings

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.preprocessing import LabelEncoder

def train_classifier(database, method="svm"):
    """Train a classifier (SVM or KNN) on face embeddings."""
    X = []
    y = []
    for person in database:
        for enc in person["embeddings"]:
            X.append(enc / np.linalg.norm(enc))  # normalize
            y.append(person["name"])

    X = np.array(X)
    y = np.array(y)

    print(f"Training data: {X.shape[0]} samples, {len(set(y))} classes")

    if X.shape[0] < 2:
        print("Need at least 2 samples to train!")
        return None, None

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    if method == "svm":
        clf = SVC(kernel="rbf", probability=True, C=1.0, gamma="scale")
        print("Training SVM classifier...")
    else:
        n_neighbors = min(3, X.shape[0])
        clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance")
        print(f"Training KNN classifier (k={n_neighbors})...")

    clf.fit(X, y_encoded)

    if X.shape[0] >= 4:
        cv = LeaveOneOut() if X.shape[0] <= 20 else 5
        scores = cross_val_score(clf, X, y_encoded, cv=cv)
        print(f"Cross-validation accuracy: {scores.mean():.1%} (+/- {scores.std():.1%})")

    model_path = os.path.join(config.MODELS_DIR, f"face_classifier_{method}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump({"classifier": clf, "label_encoder": le}, f)
    print(f"Model saved: {model_path}")

    return clf, le

if len(database) >= 2 and sum(len(p["embeddings"]) for p in database) >= 4:
    print("=== Training SVM Classifier ===")
    svm_clf, svm_le = train_classifier(database, method="svm")
    print("\n" + "="*50 + "\n")
    print("=== Training KNN Classifier ===")
    knn_clf, knn_le = train_classifier(database, method="knn")
else:
    print("Need >= 2 persons and >= 4 total images to train classifier.")
    print("Cosine similarity method still works fine.")

## 8. Quick Test with a New Image

In [ ]:
# === Test recognition with a new image ===
# Change the test image path here:
TEST_IMAGE_PATH = ""  # e.g. "test_images/test1.jpg"

if TEST_IMAGE_PATH and os.path.exists(TEST_IMAGE_PATH):
    test_image = cv2.imread(TEST_IMAGE_PATH)

    # Detect faces
    faces = app.get(test_image)
    print(f"Found {len(faces)} faces in test image")

    # Load database
    recognizer = FaceRecognizer(config.EMBEDDINGS_FILE, threshold=optimal_threshold)

    # Display results
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    img_rgb = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    for face in faces:
        x1, y1, x2, y2 = face.bbox.astype(int)
        embedding = face.embedding

        name, similarity, person_id = recognizer.match(embedding)

        color = "red" if name else "gray"
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)

        if name:
            ax.text(x1, y1 - 5, f"MISSING: {name} ({similarity:.0%})",
                    color="red", fontsize=10, fontweight="bold",
                    bbox=dict(boxstyle="round", facecolor="yellow", alpha=0.8))
            print(f"  MATCH: {name} (similarity={similarity:.3f})")
        else:
            ax.text(x1, y1 - 5, f"Unknown (sim={similarity:.2f})",
                    color="gray", fontsize=8)
            print(f"  No match (max similarity={similarity:.3f})")

    ax.axis("off")
    ax.set_title("Recognition Results on Test Image")
    plt.tight_layout()
    plt.show()
else:
    print("No test image specified. Set TEST_IMAGE_PATH above to test.")
    print("Or run the full pipeline:")
    print("  python detect_missing_person.py --video path/to/video.mp4")